# Intrusion Detection on CIC-IDS2017 Using Transformers

This notebook implements a **Transformer-based** approach for network intrusion detection using the CIC-IDS2017 dataset.
It follows the same data loading and preprocessing pipeline as the reference notebook
(`Intrusion-Detection-CIC-IDS2017.ipynb`) and replaces traditional ML classifiers with a custom
Transformer Encoder model built in PyTorch.

**Highlights**
- Same dataset cleaning, feature engineering, and balancing strategy as the reference notebook.
- A lightweight Transformer Encoder adapted for tabular / network-flow data.
- Both **binary** (Benign vs Attack) and **multi-class** (attack-type) classification.
- Evaluation with confusion matrices, classification reports, accuracy, and ROC / Precision-Recall curves.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
sns.set(style='darkgrid')
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import IncrementalPCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, classification_report, accuracy_score,
    roc_auc_score, roc_curve, auc, precision_recall_curve
)
from imblearn.over_sampling import SMOTE

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

## 2. Load the CIC-IDS2017 Dataset

Update the `DATA_DIR` variable below to point to the folder containing the eight CSV files from the CIC-IDS2017 dataset.

In [ ]:
# ---- Configure your data directory here ----
DATA_DIR = '/content/drive/MyDrive/MachineLearningCVE/'

csv_files = [
    'Monday-WorkingHours.pcap_ISCX.csv',
    'Tuesday-WorkingHours.pcap_ISCX.csv',
    'Wednesday-workingHours.pcap_ISCX.csv',
    'Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv',
    'Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv',
    'Friday-WorkingHours-Morning.pcap_ISCX.csv',
    'Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv',
    'Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv',
]

data_list = [pd.read_csv(DATA_DIR + f) for f in csv_files]

print('Data dimensions:')
for i, df in enumerate(data_list, start=1):
    print(f'  Data{i} -> {df.shape[0]} rows, {df.shape[1]} columns')

In [ ]:
data = pd.concat(data_list, ignore_index=True)
for d in data_list:
    del d

print(f'Combined shape: {data.shape}')

## 3. Data Preprocessing

In [ ]:
# Strip whitespace from column names
data.columns = data.columns.str.strip()

# Drop duplicates
before = len(data)
data.drop_duplicates(inplace=True)
print(f'Dropped {before - len(data)} duplicate rows  ->  {len(data)} remaining')

In [ ]:
# Replace infinite values with NaN, then fill with median
data.replace([np.inf, -np.inf], np.nan, inplace=True)

for col in data.select_dtypes(include=[np.number]).columns:
    if data[col].isna().any():
        data[col].fillna(data[col].median(), inplace=True)

print(f'Remaining NaNs: {data.isna().sum().sum()}')

In [ ]:
# Map labels to attack types (same mapping as reference notebook)
attack_map = {
    'BENIGN': 'BENIGN',
    'DDoS': 'DDoS',
    'DoS Hulk': 'DoS',
    'DoS GoldenEye': 'DoS',
    'DoS slowloris': 'DoS',
    'DoS Slowhttptest': 'DoS',
    'PortScan': 'Port Scan',
    'FTP-Patator': 'Brute Force',
    'SSH-Patator': 'Brute Force',
    'Bot': 'Bot',
    'Web Attack \xe2\x80\x93 Brute Force': 'Web Attack',
    'Web Attack \xe2\x80\x93 XSS': 'Web Attack',
    'Web Attack \xe2\x80\x93 Sql Injection': 'Web Attack',
    'Infiltration': 'Infiltration',
    'Heartbleed': 'Heartbleed',
}

data['Attack Type'] = data['Label'].map(attack_map)
data.drop('Label', axis=1, inplace=True)

print(data['Attack Type'].value_counts())

In [ ]:
# Memory optimization: downcast numeric types
for col in data.select_dtypes(include=[np.number]).columns:
    c_min, c_max = data[col].min(), data[col].max()
    if np.issubdtype(data[col].dtype, np.floating) and c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
        data[col] = data[col].astype(np.float32)
    elif np.issubdtype(data[col].dtype, np.integer) and c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
        data[col] = data[col].astype(np.int32)

# Drop columns with zero variance
num_unique = data.nunique()
data = data[num_unique[num_unique > 1].index]

print(f'Shape after dropping zero-variance columns: {data.shape}')

## 4. Feature Scaling & PCA

In [ ]:
features = data.drop('Attack Type', axis=1)
attacks = data['Attack Type']

scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

# Incremental PCA (same as reference notebook)
n_components = len(features.columns) // 2
ipca = IncrementalPCA(n_components=n_components, batch_size=500)
for batch in np.array_split(scaled_features, max(1, len(features) // 500)):
    ipca.partial_fit(batch)

print(f'Information retained after PCA: {sum(ipca.explained_variance_ratio_):.2%}')

transformed = ipca.transform(scaled_features)
new_data = pd.DataFrame(transformed, columns=[f'PC{i+1}' for i in range(n_components)])
new_data['Attack Type'] = attacks.values

print(f'Transformed data shape: {new_data.shape}')

## 5. Transformer Encoder Model

We define a lightweight **Transformer Encoder** designed for tabular data. Each input feature is
projected into a `d_model`-dimensional embedding, then processed by a stack of Transformer Encoder
layers. A learnable `[CLS]` token aggregates information across all features, and a classification
head maps it to the target classes.

In [ ]:
class TabularTransformer(nn.Module):
    """Transformer Encoder for tabular / network-flow classification."""

    def __init__(self, input_dim, num_classes, d_model=64, nhead=4,
                 num_layers=2, dim_feedforward=128, dropout=0.1):
        super().__init__()
        # Project each scalar feature into d_model dimensions
        self.feature_embedding = nn.Linear(1, d_model)
        # Learnable [CLS] token
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        # Positional encoding (learnable)
        self.pos_embedding = nn.Parameter(torch.randn(1, input_dim + 1, d_model))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.layer_norm = nn.LayerNorm(d_model)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, num_classes),
        )

    def forward(self, x):
        # x shape: (batch, input_dim)
        batch_size = x.size(0)
        # (batch, input_dim, 1) -> (batch, input_dim, d_model)
        x = self.feature_embedding(x.unsqueeze(-1))
        # Prepend [CLS] token
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)  # (batch, input_dim+1, d_model)
        x = x + self.pos_embedding[:, :x.size(1), :]
        x = self.transformer_encoder(x)
        x = self.layer_norm(x[:, 0, :])  # [CLS] output
        return self.classifier(x)

## 6. Training Utilities

In [ ]:
def create_dataloaders(X_train, y_train, X_test, y_test, batch_size=256):
    """Convert numpy arrays into PyTorch DataLoaders."""
    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.long)
    X_test_t = torch.tensor(X_test, dtype=torch.float32)
    y_test_t = torch.tensor(y_test, dtype=torch.long)

    train_loader = DataLoader(
        TensorDataset(X_train_t, y_train_t),
        batch_size=batch_size, shuffle=True
    )
    test_loader = DataLoader(
        TensorDataset(X_test_t, y_test_t),
        batch_size=batch_size, shuffle=False
    )
    return train_loader, test_loader

In [ ]:
def train_model(model, train_loader, num_epochs=20, lr=1e-3):
    """Train the transformer model and return loss history."""
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.5)

    model.to(DEVICE)
    history = {'train_loss': [], 'train_acc': []}

    for epoch in range(num_epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * X_batch.size(0)
            correct += (outputs.argmax(dim=1) == y_batch).sum().item()
            total += y_batch.size(0)

        scheduler.step()
        epoch_loss = running_loss / total
        epoch_acc = correct / total
        history['train_loss'].append(epoch_loss)
        history['train_acc'].append(epoch_acc)
        print(f'Epoch {epoch+1:>2}/{num_epochs} - loss: {epoch_loss:.4f} - acc: {epoch_acc:.4f}')

    return history

In [ ]:
def evaluate_model(model, test_loader):
    """Return predictions and true labels from the test set."""
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(DEVICE)
            outputs = model(X_batch)
            probs = torch.softmax(outputs, dim=1).cpu().numpy()
            preds = outputs.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(y_batch.numpy())
            all_probs.append(probs)
    return np.array(all_preds), np.array(all_labels), np.vstack(all_probs)

## 7. Visualization Helpers

In [ ]:
def plot_training_history(history, title='Training History'):
    """Plot loss and accuracy curves."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(history['train_loss'])
    axes[0].set_title(f'{title} - Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')

    axes[1].plot(history['train_acc'])
    axes[1].set_title(f'{title} - Accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_confusion_matrix(y_true, y_pred, class_names, title='Confusion Matrix'):
    """Plot a confusion matrix heatmap."""
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(max(6, len(class_names)), max(5, len(class_names) - 1)))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title(title)
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_roc_curve(y_true, y_prob, title='ROC Curve'):
    """Plot ROC curve for binary classification."""
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(7, 5))
    plt.plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc:.4f})')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(title)
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_precision_recall(y_true, y_prob, title='Precision-Recall Curve'):
    """Plot Precision-Recall curve for binary classification."""
    precision, recall, _ = precision_recall_curve(y_true, y_prob)

    plt.figure(figsize=(7, 5))
    plt.plot(recall, precision)
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title(title)
    plt.tight_layout()
    plt.show()

## 8. Binary Classification – Benign vs Attack

Following the reference notebook, we balance benign and attack traffic, then sample 15 000 rows.

In [ ]:
# Balance benign vs attack and sample
normal_traffic = new_data.loc[new_data['Attack Type'] == 'BENIGN']
intrusions = new_data.loc[new_data['Attack Type'] != 'BENIGN']
normal_traffic = normal_traffic.sample(n=min(len(normal_traffic), len(intrusions)), replace=False, random_state=0)

ids_data = pd.concat([intrusions, normal_traffic])
ids_data['Attack Type'] = np.where(ids_data['Attack Type'] == 'BENIGN', 0, 1)
bc_data = ids_data.sample(n=15000, random_state=0)

print(bc_data['Attack Type'].value_counts())

In [ ]:
X_bc = bc_data.drop('Attack Type', axis=1).values
y_bc = bc_data['Attack Type'].values

X_train_bc, X_test_bc, y_train_bc, y_test_bc = train_test_split(
    X_bc, y_bc, test_size=0.25, random_state=0
)

train_loader_bc, test_loader_bc = create_dataloaders(X_train_bc, y_train_bc, X_test_bc, y_test_bc)
print(f'Train size: {len(X_train_bc)}, Test size: {len(X_test_bc)}')

In [ ]:
bc_model = TabularTransformer(
    input_dim=X_train_bc.shape[1],
    num_classes=2,
    d_model=64,
    nhead=4,
    num_layers=2,
    dim_feedforward=128,
    dropout=0.1,
)
print(bc_model)
print(f'\nTotal parameters: {sum(p.numel() for p in bc_model.parameters()):,}')

In [ ]:
bc_history = train_model(bc_model, train_loader_bc, num_epochs=20, lr=1e-3)

In [ ]:
plot_training_history(bc_history, title='Binary Classification')

## 9. Binary Classification – Evaluation

In [ ]:
y_pred_bc, y_true_bc, y_probs_bc = evaluate_model(bc_model, test_loader_bc)

acc_bc = accuracy_score(y_true_bc, y_pred_bc)
print(f'Binary Classification Accuracy: {acc_bc:.4f}')
print()
print(classification_report(y_true_bc, y_pred_bc, target_names=['Benign', 'Attack']))

In [ ]:
plot_confusion_matrix(y_true_bc, y_pred_bc, class_names=['Benign', 'Attack'],
                      title='Binary Classification - Confusion Matrix')

In [ ]:
# ROC Curve (use probability of class 1 = Attack)
plot_roc_curve(y_true_bc, y_probs_bc[:, 1],
              title='Binary Classification - ROC Curve')

In [ ]:
# Precision-Recall Curve
plot_precision_recall(y_true_bc, y_probs_bc[:, 1],
                     title='Binary Classification - Precision-Recall Curve')

## 10. Multi-class Classification – Attack Type Detection

Following the reference notebook, we select attack types with enough samples, apply SMOTE for
class balancing, and train a Transformer model.

In [ ]:
class_counts = new_data['Attack Type'].value_counts()
selected_classes = class_counts[class_counts > 1950]
class_names = selected_classes.index
selected = new_data[new_data['Attack Type'].isin(class_names)]

dfs = []
for name in class_names:
    df = selected[selected['Attack Type'] == name]
    if len(df) > 5000:
        df = df.sample(n=5000, random_state=0)
    dfs.append(df)

df = pd.concat(dfs, ignore_index=True)
print(df['Attack Type'].value_counts())

In [ ]:
X_mc = df.drop('Attack Type', axis=1)
y_mc = df['Attack Type']

smote = SMOTE(sampling_strategy='auto', random_state=0)
X_upsampled, y_upsampled = smote.fit_resample(X_mc, y_mc)

blnc_data = pd.DataFrame(X_upsampled)
blnc_data['Attack Type'] = y_upsampled
blnc_data = blnc_data.sample(frac=1, random_state=0)

print(blnc_data['Attack Type'].value_counts())

In [ ]:
# Encode labels
le = LabelEncoder()
mc_features = blnc_data.drop('Attack Type', axis=1).values
mc_labels = le.fit_transform(blnc_data['Attack Type'].values)
num_classes = len(le.classes_)

print(f'Classes ({num_classes}):')
for i, name in enumerate(le.classes_):
    print(f'  {i}: {name}')

X_train_mc, X_test_mc, y_train_mc, y_test_mc = train_test_split(
    mc_features, mc_labels, test_size=0.25, random_state=0
)

train_loader_mc, test_loader_mc = create_dataloaders(X_train_mc, y_train_mc, X_test_mc, y_test_mc)
print(f'\nTrain size: {len(X_train_mc)}, Test size: {len(X_test_mc)}')

In [ ]:
mc_model = TabularTransformer(
    input_dim=X_train_mc.shape[1],
    num_classes=num_classes,
    d_model=64,
    nhead=4,
    num_layers=3,
    dim_feedforward=128,
    dropout=0.1,
)
print(mc_model)
print(f'\nTotal parameters: {sum(p.numel() for p in mc_model.parameters()):,}')

In [ ]:
mc_history = train_model(mc_model, train_loader_mc, num_epochs=25, lr=1e-3)

In [ ]:
plot_training_history(mc_history, title='Multi-class Classification')

## 11. Multi-class Classification – Evaluation

In [ ]:
y_pred_mc, y_true_mc, y_probs_mc = evaluate_model(mc_model, test_loader_mc)

acc_mc = accuracy_score(y_true_mc, y_pred_mc)
print(f'Multi-class Classification Accuracy: {acc_mc:.4f}')
print()
print(classification_report(y_true_mc, y_pred_mc, target_names=le.classes_))

In [ ]:
plot_confusion_matrix(y_true_mc, y_pred_mc, class_names=le.classes_,
                      title='Multi-class Classification - Confusion Matrix')

In [ ]:
# Classification report heatmap
report = classification_report(y_true_mc, y_pred_mc, target_names=le.classes_, output_dict=True)
precision_vals = [report[c]['precision'] for c in le.classes_]
recall_vals = [report[c]['recall'] for c in le.classes_]
f1_vals = [report[c]['f1-score'] for c in le.classes_]

report_data = np.array([precision_vals, recall_vals, f1_vals])
rows_labels = ['Precision', 'Recall', 'F1-score']

plt.figure(figsize=(max(10, len(le.classes_) * 2), 4))
sns.heatmap(report_data, cmap='Pastel1', annot=True, fmt='.2f',
            xticklabels=le.classes_, yticklabels=rows_labels)
plt.title('Multi-class Classification Report (Transformer)')
plt.tight_layout()
plt.show()

## 12. Summary

In [ ]:
palette = sns.color_palette('Blues', n_colors=2)

labels = ['Binary (Benign vs Attack)', 'Multi-class (Attack Types)']
scores = [acc_bc, acc_mc]

fig, ax = plt.subplots(figsize=(9, 3))
ax.barh(labels, scores, color=palette)
ax.set_xlim([0, 1])
ax.set_xlabel('Accuracy Score')
ax.set_title('Transformer Model - Accuracy Comparison')

for i, v in enumerate(scores):
    ax.text(v + 0.01, i, f'{v:.4f}', ha='left', va='center')

plt.tight_layout()
plt.show()

### Conclusions

- The **Transformer Encoder** successfully learns to classify network traffic from the CIC-IDS2017 dataset.
- For **binary classification**, the model distinguishes benign from malicious traffic with high accuracy.
- For **multi-class classification**, the model identifies specific attack types after SMOTE-based balancing.
- The self-attention mechanism in the Transformer allows the model to capture complex interactions between
  network flow features, which can be advantageous over traditional ML models for nuanced attack patterns.
- Further improvements could include hyperparameter tuning, deeper architectures, or pre-training strategies.